<a href="https://colab.research.google.com/github/tensorbytes0202/Deep-learning/blob/main/Floor_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/floor_detection.zip"
extract_path = "/content/floor_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted!")

Dataset Extracted!


In [14]:
import os
import cv2
import numpy as np

mask_dir = "/content/floor_dataset/X"
image_dir = "/content/floor_dataset/Y"

output_label_dir = "/content/labels_seg"
os.makedirs(output_label_dir, exist_ok=True)

# class mapping (adjust according to dataset)
CLASS_MAP = {
    1: 0,   # door
    2: 1    # window
}

def mask_to_yolo_seg(mask_path, output_path):
    mask = cv2.imread(mask_path, 0)
    h, w = mask.shape

    lines = []

    unique_vals = np.unique(mask)

    for val in unique_vals:
        if val not in CLASS_MAP:
            continue

        binary = (mask == val).astype(np.uint8)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            if len(cnt) < 3:
                continue

            cnt = cnt.squeeze()

            # normalize points
            polygon = []
            for x, y in cnt:
                polygon.append(x / w)
                polygon.append(y / h)

            class_id = CLASS_MAP[val]

            line = f"{class_id} " + " ".join(map(str, polygon))
            lines.append(line)

    with open(output_path, "w") as f:
        f.write("\n".join(lines))


# run for all files
for file in os.listdir(mask_dir):
    if file.endswith(".png"):
        mask_path = os.path.join(mask_dir, file)
        txt_name = file.replace(".png", ".txt")
        output_path = os.path.join(output_label_dir, txt_name)

        mask_to_yolo_seg(mask_path, output_path)

print("✅ Segmentation labels created!")

✅ Segmentation labels created!


In [16]:
from sklearn.model_selection import train_test_split
import shutil
import os

images = os.listdir(image_dir)

train, val = train_test_split(images, test_size=0.2, random_state=42)

base = "/content/dataset_seg"

for p in [
    "images/train", "images/val",
    "labels/train", "labels/val"
]:
    os.makedirs(os.path.join(base, p), exist_ok=True)

def move(files, img_dst, lbl_dst):
    for f in files:
        # Get the base name of the image file without extension
        image_base_name = os.path.splitext(f)[0]

        # Determine the corresponding label file name.
        # If the image_base_name contains ' (2)', remove it to match the generated label file names.
        if image_base_name.endswith(' (2)'):
            label_file_base_name = image_base_name[:-4] # Remove ' (2)'
        else:
            label_file_base_name = image_base_name

        # Copy the image file
        shutil.copy(os.path.join(image_dir, f), os.path.join(img_dst, f))

        # Copy the corresponding label file
        shutil.copy(os.path.join(output_label_dir, label_file_base_name + ".txt"),
                    os.path.join(lbl_dst, label_file_base_name + ".txt"))

move(train, base+"/images/train", base+"/labels/train")
move(val, base+"/images/val", base+"/labels/val")

In [17]:
data_yaml = """
train: /content/dataset_seg/images/train
val: /content/dataset_seg/images/val

nc: 2
names: ['door', 'window']
"""

with open("/content/dataset_seg/data.yaml", "w") as f:
    f.write(data_yaml)

In [20]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

model.train(
    data="/content/dataset_seg/data.yaml",
    epochs=50,
    imgsz=640
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_seg/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscrip

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/50      2.62G          0          0      127.1          0          0          0        640: 50% ━━━━━━────── 3/6 1.6s/it 5.9s<4.9s


Exception ignored in: <generator object TQDM.__iter__ at 0x7d1e994aa4d0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/tqdm.py", line 354, in __iter__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/tqdm.py", line 332, in close
    self.file.flush()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 488, in flush
    if not evt.wait(self.flush_timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/threading.py", line 655, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/threading.py", line 359, in wait
    gotit = waiter.acquire(True, timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


KeyboardInterrupt: 